In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
train = pd.read_csv("./house_prices/train.csv")
test = pd.read_csv("./house_prices/test.csv")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
train.head()

In [ ]:
train['Fence'].isnull().sum()

In [ ]:
train.info()


In [ ]:
train.describe()

In [ ]:
train.isnull().sum()

Note : NaN values doesnt get counted in isnull() function

In [ ]:
# perecentage of missing values
train.isnull().mean() * 100

In [ ]:
# total n.o of missing values in each column
train.isnull().sum().sort_values(ascending=False).head(12)

In [ ]:
# missing values summary

# table
missing_counts = train.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
missing_pct = (missing_counts / len(train)) * 100
missing_df = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})

print("Columns with missing values:\n")
print(missing_df.head(15))

# plot
plt.figure(figsize=(10,5))
sns.barplot(x=missing_df.index, y=missing_df.missing_pct, color="steelblue")
plt.xticks(rotation=90)
plt.ylabel("% missing")
plt.title("Missing values per column (%)")
plt.show()


Note : Columns like PoolQC, Alley, and MiscFeature have >95% missing values, meaning they provide little value. I decided to drop them. For Fence (80% missing), I may keep it by treating “missing” as “no fence,” but dropping is also acceptable depending on downstream modeling needs.

In [ ]:
columns_to_drop = ['PoolQC' ,'Alley' ,'MiscFeature']
# training data
train = train.drop(columns=columns_to_drop , axis = 1)
# testing data
test = test.drop(columns=columns_to_drop , axis = 1)
print("Remaining cols after drop :" ,train.shape[1])

Note : I dropped PoolQC, Alley, and MiscFeature since they had >95% missing values and were unlikely to add predictive value.

In [ ]:
# Categorical features where NA means "None"
cat_fill_none = [
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'MasVnrType'
]

for col in cat_fill_none:
    train[col] = train[col].fillna("None")
    test[col]  = test[col].fillna("None")

# Numerical features where NA means 0 (not present)
num_fill_zero = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea'
]

for col in num_fill_zero:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)


Note :  I treated missing values in garage, basement, fireplace, and masonry features as structural absence. Categorical fields (e.g., GarageType, BsmtQual) were filled with "None", while numeric fields (e.g., MasVnrArea, BsmtFinSF1) were filled with 0. This preserves the true meaning of “not present.”

In [ ]:
train['LotFrontage']

In [ ]:
train['LotFrontage'] =train.groupby('Neighborhood')['LotFrontage'].transform(lambda x : x.fillna(x.median()))
test['LotFrontage'] = test.groupby('Neighborhood')['LotFrontage'].transform(lambda x : x.fillna(x.median()))

Note : Since lot sizes are influenced by neighborhood planning, I imputed missing LotFrontage values with the median frontage of the respective Neighborhood. This preserves locality patterns and avoids bias from using a single global statistic.

In [ ]:
train['Fence']=  train['Fence'].fillna('None')
test['Fence']=  test['Fence'].fillna('None')

In [ ]:
# checking back again , if any missing values are left
print(train.isnull().sum().sort_values(ascending = False).head(10))
print(test.isnull().sum().sort_values(ascending = False).head(10))

In [ ]:
# fill Electrical in train
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])

# Fill categorical vars in test with mode
for col in ['MSZoning', 'Utilities', 'Functional', 'Exterior1st',
            'Exterior2nd', 'KitchenQual', 'SaleType']:
    test[col] = test[col].fillna(test[col].mode()[0])


Note : I imputed the few remaining categorical missing values with their respective most frequent category (mode). This ensures minimal distortion, since the percentage of missingness was tiny (<0.5%).

In [ ]:
# checking back again , if any missing values are left
print(train.isnull().sum().sort_values(ascending = False).head(10))
print(test.isnull().sum().sort_values(ascending = False).head(10))